[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C59_VLA_Perception_Interface_Course/04_instruction/04_traffic_rule_following.ipynb)

# 04 · 交通规则的指令跟随与约束化（标志→约束 / 生命周期 / 冲突消解 / 安全兜底 / 快慢双系统）

目标：把「VLA 看到限速牌会自己减速」这句话，拆成**四个可运行、可单测、可回归的组件**。

本 notebook 你会亲手实现：
1. **标志 → 可执行约束的翻译器**（三级翻译：感知语义 → 规范语义 → 约束对象，含辅助牌限定符）
2. **约束的生命周期状态机**（TENTATIVE→CONFIRMED→ACTIVE→EXPIRED/REVOKED/SUPERSEDED），
   并用它复现**「匝道限速带上高速」**这个经典 bug，量化 over-hold 里程
3. **冲突消解**：字典序三元组 (authority, specificity, recency) + 平票兜底 + **「最保守」的反例**
4. **不可投影约束**：禁左的轨迹候选屏蔽、停车让行的时空判定（含 rolling stop 检测）
5. **安全兜底层**：五项校验、三态输出 ACCEPT / CLIP / REJECT，以及**干预率**统计
6. **快慢双系统**：从物理量倒推可用推理时间，算出「这一帧最多能跑几个 CoT token」

> 心智模型：**感知回答「有什么」，约束层回答「因此不能做什么」，VLA 回答「在能做的里面选哪个」。**
> 三件事混在一个网络里，出问题时你连该修哪一段都不知道。

## 1 · 三级翻译：从检测框到可执行约束

TSR 给的是**观测**（类别 + 置信度 + 测距 + 关联车道 + 跟踪 ID），不是指令。
中间必须显式经过「规范语义」这一层，产出一个带**类型 / 值 / 作用域 / 来源 / 权威度 / 状态 / 证据链**的对象。

注意一个细节：约束的生效里程是 **`ego_s + range_m`（牌所在里程）**，
不是「检出时的自车里程」——这两个差着几十米，写错就是提前生效或提前失效。

In [ ]:
import numpy as np, math

KMH = 1.0 / 3.6                    # km/h -> m/s

# ── 来源权威度（第 3 节消解用；数字越大越优先）──
AUTHORITY = {'POLICE': 5, 'TEMPORARY': 4, 'VMS': 3, 'FIXED_SIGN': 2, 'HDMAP': 1, 'DEFAULT': 0}

# ── 标志类别 -> 规范语义 ──
#    end_rules   : 哪些事件会让这条约束失效（**少写一条就是一整类 bug**）
#    projectable : 违反时能否投影修复（决定安全层是 CLIP 还是 REJECT）
#    cons        : 保守方向（MIN / MAX / UNION）—— 不能一律取 min，见第 3 节反例
SIGN_TABLE = {
    'speed_limit':     dict(kind='SPEED_MAX',    arg=None,       projectable=True,
                            end_rules=('END_SIGN', 'JUNCTION', 'ROAD_CLASS_CHANGE'), cons='MIN'),
    'min_speed':       dict(kind='SPEED_MIN',    arg=None,       projectable=True,
                            end_rules=('END_SIGN', 'JUNCTION'),                      cons='MAX'),
    'no_left_turn':    dict(kind='BAN_MANEUVER', arg='left',     projectable=False,
                            end_rules=('JUNCTION_PASSED',),                          cons='UNION'),
    'no_uturn':        dict(kind='BAN_MANEUVER', arg='uturn',    projectable=False,
                            end_rules=('JUNCTION_PASSED',),                          cons='UNION'),
    'no_overtake':     dict(kind='BAN_MANEUVER', arg='overtake', projectable=False,
                            end_rules=('END_SIGN',),                                 cons='UNION'),
    'stop':            dict(kind='FULL_STOP',    arg=None,       projectable=False,
                            end_rules=(),                                            cons='UNION'),
    'end_speed_limit': dict(kind='REVOKE',       arg=None,       projectable=None,
                            end_rules=(), cons=None, target='SPEED_MAX'),
    'end_no_overtake': dict(kind='REVOKE',       arg=None,       projectable=None,
                            end_rules=(), cons=None, target='BAN_MANEUVER:overtake'),
}

def translate(det, ego_s, t):
    """① 感知语义 -> ③ 可执行约束。det 是模块 03 的 TSR 序列化 schema 的一条。"""
    spec  = SIGN_TABLE[det['sign']]
    plate = det.get('plate', {})                    # 辅助牌 = 约束的限定符
    begins = ego_s + det['range_m']                 # **牌所在里程**，不是检出时的自车里程
    span   = plate.get('range_m')                   # 辅助牌「前方 2 km」
    src    = det.get('source', 'FIXED_SIGN')
    return dict(
        id=None, kind=spec['kind'], arg=spec['arg'], target=spec.get('target'),
        value=(det['value'] * KMH if spec['kind'] in ('SPEED_MAX', 'SPEED_MIN') else None),
        source=src, authority=AUTHORITY[src],
        lanes=tuple(plate.get('lanes', ('ego',))),
        vehicle_class=plate.get('vehicle_class', 'all'),
        time_window=tuple(plate.get('time_window', (0.0, 24.0))),
        begins_at_s=begins,
        ends_at_s=(begins + span) if span else float('inf'),
        end_rules=spec['end_rules'], projectable=spec['projectable'],
        conservatism=spec['cons'],
        specificity=len(plate),                     # 限定符越多 -> 越具体（lex specialis）
        t_obs=t, conf=det['conf'], evidence=[det['track_id']],
        state='TENTATIVE', hits=1, misses=0,
    )

# —— 例 1：一块普通的限速 60 牌，82.4 m 外 ——
d60 = dict(sign='speed_limit', value=60, conf=0.93, range_m=82.4,
           track_id=47, source='FIXED_SIGN')
c = translate(d60, ego_s=1157.6, t=12.4)
print('kind=%s  value=%.3f m/s (%.0f km/h)  scope_s=[%.1f, %s)  lanes=%s'
      % (c['kind'], c['value'], c['value'] / KMH, c['begins_at_s'],
         c['ends_at_s'], c['lanes']))
print('   authority=%d  specificity=%d  projectable=%s  conservatism=%s'
      % (c['authority'], c['specificity'], c['projectable'], c['conservatism']))
assert abs(c['value'] - 60 * KMH) < 1e-12
assert abs(c['begins_at_s'] - 1240.0) < 1e-9, '生效里程 = 自车里程 + 测距'
assert c['ends_at_s'] == float('inf') and c['projectable'] is True

# —— 例 2：带三个辅助牌限定符的临时施工限速 40 ——
d40 = dict(sign='speed_limit', value=40, conf=0.81, range_m=55.0, track_id=48,
           source='TEMPORARY',
           plate=dict(lanes=('ego', 'right'), vehicle_class='truck', range_m=2000.0))
c2 = translate(d40, ego_s=1200.0, t=13.0)
print('\n临时施工牌：authority=%d（TEMPORARY）specificity=%d  scope_s=[%.0f, %.0f)  车型=%s'
      % (c2['authority'], c2['specificity'], c2['begins_at_s'], c2['ends_at_s'], c2['vehicle_class']))
assert c2['authority'] == 4 and c2['specificity'] == 3
assert c2['ends_at_s'] == 1255.0 + 2000.0

# —— 例 3：禁止左转（不可投影！）——
c3 = translate(dict(sign='no_left_turn', conf=0.88, range_m=30.0, track_id=49), 500.0, 20.0)
print('禁左：kind=%s arg=%s **projectable=%s** -> 违反时只能整条 REJECT'
      % (c3['kind'], c3['arg'], c3['projectable']))
assert c3['kind'] == 'BAN_MANEUVER' and c3['arg'] == 'left' and c3['projectable'] is False
print('\n✅ 翻译器就位：约束是对象，不是一个数。')

## 2 · 生命周期状态机：约束什么时候生效，什么时候死

两条最容易被合并、但必须分开的状态：
- **CONFIRMED** = 「我确认前方 80 m 有一块限速 60 的牌」
- **ACTIVE**    = 「我现在应该以 60 行驶」

中间隔着 80 米。合并了，车就会**一看见牌就减速**——用户会直接感知为「这车反应很奇怪」。

失效有五条路：显式解除 / 路口隐式失效 / 道路等级变化 / 范围耗尽 / 被取代。
下面把它们全部实现，并加一条**运行时不变式**：
同一 `(kind, arg, lanes, source)` 槽位上最多一条 ACTIVE 约束（跨来源并存是正常的，交给第 3 节消解）。

In [ ]:
def _match_target(c, target):
    """解除牌的匹配：'SPEED_MAX' 或 'BAN_MANEUVER:overtake'"""
    if not target:
        return False
    if ':' in target:
        k, a = target.split(':')
        return c['kind'] == k and c['arg'] == a
    return c['kind'] == target

def _slot(c):
    """约束的「槽位」：同槽位不允许两条 ACTIVE 并存。"""
    return (c['kind'], c['arg'], frozenset(c['lanes']), c['source'])


class ConstraintManager:
    """约束的生命周期管理。所有状态转移集中在 step() 里 —— 便于单测与日志回放。"""

    def __init__(self, confirm_hits=3, conf_hi=0.70, drop_after=4):
        self.confirm_hits, self.conf_hi, self.drop_after = confirm_hits, conf_hi, drop_after
        self.store = {}          # track_id -> constraint
        self.log = []            # (t, ego_s, id, from, to, why)  <- 第 5 节的 resolution_trace
        self._next_id = 1
        self._now = (0.0, 0.0)

    def _set(self, c, new, why):
        if c['state'] != new:
            t, s = self._now
            self.log.append((t, s, c['id'], c['state'], new, why))
            c['state'] = new

    def step(self, t, ego_s, dets, events=()):
        self._now = (t, ego_s)
        seen = set()
        # ① 观测融合：老 track 累加证据，新 track 建约束（TENTATIVE）
        for d in dets:
            tid = d['track_id']; seen.add(tid)
            if tid in self.store:
                c = self.store[tid]
                c['hits'] += 1; c['misses'] = 0
                c['conf'] = max(c['conf'], d['conf'])
            else:
                c = translate(d, ego_s, t)
                c['id'] = 'c%d' % self._next_id; self._next_id += 1
                self.store[tid] = c
                self.log.append((t, ego_s, c['id'], '-', 'TENTATIVE', 'first_seen'))
        for tid, c in self.store.items():
            if tid not in seen:
                c['misses'] += 1
        # ② TENTATIVE -> CONFIRMED（证据够）/ DROPPED（久未复现 = 单帧误检）
        for c in self.store.values():
            if c['state'] == 'TENTATIVE':
                if c['hits'] >= self.confirm_hits and c['conf'] >= self.conf_hi:
                    self._set(c, 'CONFIRMED', 'evidence_ok(hits=%d,conf=%.2f)' % (c['hits'], c['conf']))
                elif c['misses'] >= self.drop_after:
                    self._set(c, 'DROPPED', 'no_reobservation')
        # ③ REVOKE 算子：已确认的解除牌撤销匹配的 ACTIVE 约束
        for c in self.store.values():
            if c['kind'] == 'REVOKE' and c['state'] == 'CONFIRMED' and ego_s >= c['begins_at_s']:
                for o in self.store.values():
                    if o['state'] == 'ACTIVE' and _match_target(o, c['target']):
                        self._set(o, 'REVOKED', 'end_sign:%s' % c['id'])
                self._set(c, 'SATISFIED', 'applied')
        # ④ CONFIRMED -> ACTIVE：**车真的开过了牌**
        for c in self.store.values():
            if c['state'] == 'CONFIRMED' and c['kind'] != 'REVOKE' and ego_s >= c['begins_at_s']:
                self._set(c, 'ACTIVE', 'ego_passed_sign')
        # ⑤ ACTIVE -> EXPIRED：范围耗尽 / 隐式失效事件（路口、道路等级变化）
        for c in self.store.values():
            if c['state'] != 'ACTIVE':
                continue
            if ego_s > c['ends_at_s']:
                self._set(c, 'EXPIRED', 'range_exhausted')
            else:
                for e in events:
                    if e in c['end_rules']:
                        self._set(c, 'EXPIRED', 'event:%s' % e); break
        # ⑥ ACTIVE -> SUPERSEDED：同槽位、更靠后的新约束取代旧的
        for c in list(self.store.values()):
            if c['state'] != 'ACTIVE':
                continue
            for o in self.store.values():
                if o is c or o['state'] != 'ACTIVE':
                    continue
                if _slot(o) == _slot(c) and o['begins_at_s'] > c['begins_at_s']:
                    self._set(c, 'SUPERSEDED', 'by:%s' % o['id']); break
        return self.active()

    def active(self):
        return [c for c in self.store.values() if c['state'] == 'ACTIVE']


def check_invariant(mgr):
    """运行时不变式：同一槽位最多一条 ACTIVE。最便宜的一道防线。"""
    seen = {}
    for c in mgr.active():
        k = _slot(c)
        assert k not in seen, ('同槽位出现两条 ACTIVE 约束', k, seen[k], c['id'])
        seen[k] = c['id']
    return len(seen)

print('✅ 状态机就位：8 个状态、6 组转移、1 条运行时不变式。')

### 2.1 复现经典 bug：「匝道限速带上高速」

场景（自车 15 m/s 匀速，dt = 0.5 s）：
- `t=1.0–2.5 s`：检出匝道限速 **60** 的牌（牌在里程 55 m），连续 4 帧
- `t=5.0 s`   ：**单帧误检**（广告牌被当成限速 30，conf 0.52）—— 应当被状态机挡掉
- `t=8.0 s`   ：驶出匝道进入主路（`ROAD_CLASS_CHANGE`，**这个信号纯视觉拿不到，必须靠地图/定位**）
- `t=9.0–10.5 s`：检出主路限速 **120** 的牌（牌在里程 195 m）

跑两遍：**有地图信号** vs **没有地图信号**，量化 over-hold（应失效之后仍生效的里程）。

In [ ]:
def make_ramp_scenario(with_map_events=True):
    frames = []
    for k in range(29):                              # t = 0.0 .. 14.0
        t = 0.5 * k
        ego_s = 15.0 * t                             # 15 m/s 匀速
        dets, events = [], []
        if 1.0 <= t <= 2.5:                          # 匝道限速 60（牌在 55 m）
            dets.append(dict(sign='speed_limit', value=60, conf=0.90,
                             range_m=55.0 - ego_s, track_id=47, source='FIXED_SIGN'))
        if t == 5.0:                                 # 单帧误检：广告牌 -> 限速 30
            dets.append(dict(sign='speed_limit', value=30, conf=0.52,
                             range_m=45.0, track_id=99, source='FIXED_SIGN'))
        if 9.0 <= t <= 10.5:                         # 主路限速 120（牌在 195 m）
            dets.append(dict(sign='speed_limit', value=120, conf=0.94,
                             range_m=195.0 - ego_s, track_id=51, source='FIXED_SIGN'))
        if with_map_events and t == 8.0:
            events.append('ROAD_CLASS_CHANGE')       # 驶出匝道 -> 匝道限速应当失效
        frames.append((t, ego_s, dets, events))
    return frames

def run(frames):
    mgr, hist = ConstraintManager(), []
    for t, ego_s, dets, events in frames:
        act = mgr.step(t, ego_s, dets, events)
        check_invariant(mgr)
        vmax = min([c['value'] for c in act if c['kind'] == 'SPEED_MAX'], default=float('inf'))
        hist.append((t, ego_s, vmax))
    return mgr, hist

def deactivation(mgr, cid):
    """返回该约束**离开 ACTIVE** 时的 (ego_s, 新状态, 原因)"""
    for (t, s, i, a, b, w) in mgr.log:
        if i == cid and a == 'ACTIVE':
            return s, b, w
    return None

mgr_ok, hist_ok = run(make_ramp_scenario(True))
mgr_bad, hist_bad = run(make_ramp_scenario(False))

print('状态转移日志（有地图信号）:')
for (t, s, i, a, b, w) in mgr_ok.log:
    print('  t=%4.1f s  ego_s=%6.1f m  %-3s  %-9s -> %-10s  %s' % (t, s, i, a, b, w))

d_ok, d_bad = deactivation(mgr_ok, 'c1'), deactivation(mgr_bad, 'c1')
print('\n匝道限速 c1 的失效位置：')
print('  有地图信号: ego_s=%.1f m  ->  %s (%s)' % d_ok)
print('  无地图信号: ego_s=%.1f m  ->  %s (%s)' % d_bad)
over_hold_ok  = d_ok[0] - 120.0                     # 应当在 ego_s=120 处失效
over_hold_bad = d_bad[0] - 120.0
print('\nover-hold（应失效后仍生效的里程）: 有地图 %.1f m   **无地图 %.1f m**'
      % (over_hold_ok, over_hold_bad))
print('   -> 无地图时，车在主路上被压在 60 km/h 跑了 %.1f m / %.1f s' % (over_hold_bad, over_hold_bad / 15.0))
assert over_hold_ok == 0.0
assert over_hold_bad == 75.0, over_hold_bad
assert d_bad[1] == 'SUPERSEDED', '没有地图信号时，旧限速只能等到下一块牌把它顶掉'

# 单帧误检必须**从未**变成 ACTIVE
c2_states = [b for (t, s, i, a, b, w) in mgr_ok.log if i == 'c2']
print('\n单帧误检 c2 的状态历程:', ' -> '.join(['TENTATIVE'] + c2_states[1:]))
assert 'ACTIVE' not in c2_states, '单帧误检绝不能生效'
assert 'DROPPED' in c2_states
print('✅ 迟滞（hits>=3 且 conf>=0.70）把单帧误检挡在了 TENTATIVE，代价是 ~1.5 s 的确认延迟。')

## 3 · 冲突消解：字典序三元组 + 平票兜底

`key(c) = (authority, specificity, begins_at_s)`，取字典序最大。三个维度分别回答：
**谁说的 / 管得多具体 / 什么时候说的**。

两个必须显式处理的东西：
1. **平票**（三元组完全相同）→ 取最保守 + **计数**。平票率上升通常意味着车道关联坏了。
2. **「最保守」不是普适规则** —— 最低限速牌的保守方向是相反的。

In [ ]:
def applies(c, lane='ego', clock_h=12.0, vehicle='car'):
    """作用域筛选：横向车道 / 车型 / 时间窗（辅助牌限定符在这里生效）"""
    if lane not in c['lanes']:
        return False
    if c['vehicle_class'] not in ('all', vehicle):
        return False
    lo, hi = c['time_window']
    return lo <= clock_h < hi

def resolve(cands, kind='SPEED_MAX'):
    """返回 (winner, trace, tie)。trace 是可写进决策日志的消解过程。"""
    pool = [c for c in cands if c['kind'] == kind]
    if not pool:
        return None, [], False
    key = lambda c: (c['authority'], c['specificity'], c['begins_at_s'])
    best = max(key(c) for c in pool)
    top = [c for c in pool if key(c) == best]
    tie = len(top) > 1
    if tie:                                   # 平票 -> 按该约束自己的保守方向兜底
        pol = top[0]['conservatism']
        win = min(top, key=lambda c: c['value']) if pol == 'MIN' else max(top, key=lambda c: c['value'])
    else:
        win = top[0]
    trace = [(c['id'], key(c), 'WIN' if c is win else 'lose') for c in pool]
    return win, trace, tie

def mk(cid, sign, value, source, ego_s, range_m, plate=None, t=0.0):
    c = translate(dict(sign=sign, value=value, conf=0.9, range_m=range_m,
                       track_id=cid, source=source, plate=plate or {}), ego_s, t)
    c['id'] = cid
    return c

# —— 场景：施工临时牌 40 / 固定牌 80 / 地图 100 同时生效 ——
cands = [mk('t1', 'speed_limit',  40, 'TEMPORARY',  1000.0, 30.0),
         mk('f1', 'speed_limit',  80, 'FIXED_SIGN', 1000.0, 10.0),
         mk('h1', 'speed_limit', 100, 'HDMAP',      1000.0,  0.0)]
win, trace, tie = resolve(cands)
print('候选与消解过程：')
for cid, k, verdict in trace:
    print('  %-3s key=(auth=%d, spec=%d, s=%7.1f)  %s' % (cid, k[0], k[1], k[2], verdict))
print('-> 生效限速 %.0f km/h（来源 %s），平票=%s' % (win['value'] / KMH, win['source'], tie))
assert win['id'] == 't1' and abs(win['value'] - 40 * KMH) < 1e-12 and not tie

# —— 加一块「本车道 + 货车」的辅助牌：特异性更高，但车型不匹配 -> 对乘用车不生效 ——
truck = mk('f2', 'speed_limit', 30, 'FIXED_SIGN', 1000.0, 20.0,
           plate=dict(lanes=('ego',), vehicle_class='truck'))
pool_car = [c for c in cands + [truck] if applies(c, vehicle='car')]
w2, _, _ = resolve(pool_car)
print('\n乘用车看到的候选：%s -> 生效 %.0f km/h' % ([c['id'] for c in pool_car], w2['value'] / KMH))
assert 'f2' not in [c['id'] for c in pool_car], '货车限速牌必须被识别出来然后忽略'
assert w2['id'] == 't1'
print('✅ 辅助牌不是装饰：它决定了这条约束到底管不管你。')

In [ ]:
# —— 平票：龙门架上并排两块牌，车道关联失败 -> 三元组完全相同 ——
tie_pool = [mk('g1', 'speed_limit', 100, 'FIXED_SIGN', 2000.0, 50.0),
            mk('g2', 'speed_limit',  80, 'FIXED_SIGN', 2000.0, 50.0)]
w3, tr3, tie3 = resolve(tie_pool)
print('平票场景：key 完全相同 -> tie=%s，取最保守 -> %.0f km/h（来自 %s）'
      % (tie3, w3['value'] / KMH, w3['id']))
assert tie3 is True and w3['id'] == 'g2'
print('⚠️  **必须计数并打日志**：平票率突然上升 = 车道关联出了问题，'
      '而兜底规则会把这个故障悄悄掩盖掉。')

# —— 反例一：最低限速牌的「保守方向」是相反的 ——
mins = [mk('m1', 'min_speed', 90, 'FIXED_SIGN', 3000.0, 40.0),
        mk('m2', 'min_speed', 60, 'FIXED_SIGN', 3000.0, 40.0)]
w4, _, tie4 = resolve(mins, kind='SPEED_MIN')
naive_min = min(c['value'] for c in mins)
print('\n最低限速平票：conservatism=%s -> 取 %.0f km/h；'
      '天真地一律 min() 会得到 %.0f km/h（**方向反了，等于放松约束**）'
      % (mins[0]['conservatism'], w4['value'] / KMH, naive_min / KMH))
assert tie4 and abs(w4['value'] - 90 * KMH) < 1e-12
assert naive_min < w4['value']

# —— 反例二：施工区解除牌漏检 -> 临时约束必须有硬性里程上限 ——
def temporary_watchdog(c, ego_s, max_span=3000.0):
    """临时约束走过 max_span 仍未见解除牌 -> 自动降级到地图限速。"""
    return c['source'] == 'TEMPORARY' and (ego_s - c['begins_at_s']) > max_span

tmp = mk('t9', 'speed_limit', 40, 'TEMPORARY', 5000.0, 0.0)
print('\n临时约束看门狗：走了 2000 m -> 降级=%s；走了 3200 m -> 降级=%s'
      % (temporary_watchdog(tmp, 5000.0 + 2000.0), temporary_watchdog(tmp, 5000.0 + 3200.0)))
assert not temporary_watchdog(tmp, 7000.0) and temporary_watchdog(tmp, 8200.0)
print('✅ 「保守 = 安全」是个漂亮但危险的等式：'
      '在车流 100 km/h 的路段以 40 行驶，被追尾风险远高于超速风险。')

## 4 · 不可投影的约束：动作屏蔽与停车让行

第 2 节表里的 `projectable` 字段在这里兑现：
- **禁左**：候选轨迹集合上的**屏蔽**操作。不能「稍微不左转一点」。
- **停车让行**：一个关于整条轨迹的**时序存在量词**
  $\exists t^\star:\ v(t^\star)\!\le\!\varepsilon \wedge s(t^\star)\in[s_0-\delta,\,s_0]$。
  实现成「速度低于阈值」是量产里真实发生过的错误 —— 那叫 **rolling stop**，是违章。

In [ ]:
DT = 0.1

def make_traj(v_profile, s0=0.0, maneuver='straight'):
    v = np.asarray(v_profile, float)
    return dict(t=np.arange(len(v)) * DT, s=s0 + np.cumsum(v) * DT, v=v, maneuver=maneuver)

def maneuver_mask(candidates, cons):
    banned = {c['arg'] for c in cons if c['kind'] == 'BAN_MANEUVER'}
    return [m for m in candidates if m not in banned], banned

def full_stop_satisfied(s, v, s_stop, tol=2.0, v_eps=0.10):
    """必须**在停止线前 tol 米内**出现一次速度 <= v_eps 的时刻。"""
    s = np.asarray(s, float); v = np.asarray(v, float)
    return bool(np.any((v <= v_eps) & (s >= s_stop - tol) & (s <= s_stop)))

# —— 动作屏蔽 ——
cons_junction = [translate(dict(sign='no_left_turn', conf=0.9, range_m=20.0, track_id=1), 0.0, 0.0),
                 translate(dict(sign='no_uturn',     conf=0.9, range_m=20.0, track_id=2), 0.0, 0.0)]
allowed, banned = maneuver_mask(['straight', 'left', 'right', 'uturn'], cons_junction)
print('候选机动 %s  -屏蔽-> %s   (banned=%s)' % (['straight', 'left', 'right', 'uturn'], allowed, sorted(banned)))
assert allowed == ['straight', 'right'] and banned == {'left', 'uturn'}

# —— 停车让行：三条轨迹 ——
v_full = np.concatenate([np.linspace(12, 0, 30), np.zeros(10), np.linspace(0, 8, 20)])   # 真停
v_roll = np.concatenate([np.linspace(12, 1.2, 30), np.full(10, 1.2), np.linspace(1.2, 8, 20)])  # rolling stop
tr_full, tr_roll = make_traj(v_full), make_traj(v_roll)
S_LINE = 18.5                       # 停止线里程（真停轨迹停在 18.0 m，线前 0.5 m）

cases = [('完全停止（停在线前 0.5 m）', tr_full, S_LINE),
         ('rolling stop（最低 1.2 m/s）', tr_roll, S_LINE),
         ('停得太早（停止线在 30 m）',    tr_full, 30.0)]
print(f"\n{'轨迹':<26s}{'最低速度 m/s':>13s}{'停止时里程 m':>14s}{'判定':>10s}")
for name, tr, sl in cases:
    ok = full_stop_satisfied(tr['s'], tr['v'], sl)
    i = int(np.argmin(tr['v']))
    print('%-26s%13.2f%14.1f%10s' % (name, tr['v'][i], tr['s'][i], 'PASS' if ok else '**FAIL**'))
assert full_stop_satisfied(tr_full['s'], tr_full['v'], S_LINE)
assert not full_stop_satisfied(tr_roll['s'], tr_roll['v'], S_LINE), 'rolling stop 必须判违规'
assert not full_stop_satisfied(tr_full['s'], tr_full['v'], 30.0), '停得太早也没有在停止线前停'
print('\n✅ 「完全停止」是时序性质，不是一个阈值比较 —— 它无法靠局部修改凑出来，只能重规划。')

## 5 · 安全兜底层：五项校验，三态输出

**红线：VLA 的输出没有任何一条可以直接下发给执行器。**

| 违反的约束 | 可投影？ | 处置 |
|---|---|---|
| 速度上界、加速度/jerk | 凸区间 → 能 | **CLIP**（最小改动，语义不变） |
| 禁行机动 | 不连通集 → 不能 | **REJECT** → 规则型 fallback |
| 停车让行 | 时序存在量词 → 不能 | **REJECT** → 重规划 |
| NaN / 维度错 / 时间戳过期 | — | **REJECT**（不试图修复未知状态） |

In [ ]:
LIMITS = dict(a_max=4.0)

def speed_upper_bound(cons):
    vs = [c['value'] for c in cons if c['kind'] == 'SPEED_MAX']
    return min(vs) if vs else float('inf')

def clip_accel(v, a_max, dt):
    """沿时间轴投影到 |a| <= a_max 的可行域（逐点 clip，保证结果一定满足）。"""
    out = v.copy()
    for i in range(1, out.size):
        lo, hi = out[i - 1] - a_max * dt, out[i - 1] + a_max * dt
        out[i] = min(max(out[i], lo), hi)
    return out

def fallback_profile(v0, n, dt=DT, decel=2.5):
    """规则型兜底轨迹：保持车道 + 匀减速。REJECT 时下发的就是它。"""
    return np.maximum(v0 - decel * dt * np.arange(n), 0.0)

def safety_gate(traj, cons, limits=LIMITS, dt=DT):
    v = np.asarray(traj['v'], float).copy()
    s = np.asarray(traj['s'], float)
    reasons = []
    # ① 有效性
    if v.size == 0 or not (np.all(np.isfinite(v)) and np.all(np.isfinite(s))):
        return dict(verdict='REJECT', reasons=['invalid_output'],
                    v=fallback_profile(0.0, max(v.size, 1), dt))
    # ② 不可投影的硬约束 -> 整条拒绝
    for c in cons:
        if c['kind'] == 'BAN_MANEUVER' and traj.get('maneuver') == c['arg']:
            reasons.append('banned_maneuver:%s [%s]' % (c['arg'], c['id']))
        elif c['kind'] == 'FULL_STOP' and not full_stop_satisfied(s, v, c['begins_at_s']):
            reasons.append('full_stop_unsatisfied [%s]' % c['id'])
    if reasons:
        return dict(verdict='REJECT', reasons=reasons, v=fallback_profile(float(v[0]), v.size, dt))
    # ③ 可投影：速度上界
    vmax, clipped = speed_upper_bound(cons), False
    if np.any(v > vmax + 1e-9):
        reasons.append('speed_max: max %.2f > %.2f m/s -> CLIP' % (v.max(), vmax))
        v = np.minimum(v, vmax); clipped = True
    # ④ 可投影：动力学
    a = np.diff(v) / dt
    if a.size and np.max(np.abs(a)) > limits['a_max'] + 1e-9:
        reasons.append('accel: |a|max %.2f > %.2f m/s^2 -> CLIP' % (np.max(np.abs(a)), limits['a_max']))
        v = clip_accel(v, limits['a_max'], dt); clipped = True
    return dict(verdict='CLIP' if clipped else 'ACCEPT', reasons=reasons, v=v)

c60 = mk('c60', 'speed_limit', 60, 'FIXED_SIGN', 0.0, 0.0)
cLeft = translate(dict(sign='no_left_turn', conf=0.9, range_m=0.0, track_id=7), 0.0, 0.0); cLeft['id'] = 'cL'
cStop = translate(dict(sign='stop', conf=0.9, range_m=18.5, track_id=8), 0.0, 0.0); cStop['id'] = 'cS'

tests = [
    ('干净轨迹 54 km/h 直行',       make_traj(np.full(40, 15.0)),                    [c60]),
    ('超速 70 km/h',               make_traj(np.full(40, 70 * KMH)),                [c60]),
    ('急加速（|a|=8 m/s²）',        make_traj(np.linspace(5, 13, 11)),               [c60]),
    ('禁左路口输出左转',            make_traj(np.full(40, 12.0), maneuver='left'),   [c60, cLeft]),
    ('STOP 牌前 rolling stop',      tr_roll,                                         [cStop]),
]
print(f"{'VLA 原始输出':<26s}{'判决':>9s}   原因")
for name, tr, cons in tests:
    r = safety_gate(tr, cons)
    print('%-26s%9s   %s' % (name, r['verdict'], '; '.join(r['reasons']) or '-'))

assert safety_gate(tests[0][1], [c60])['verdict'] == 'ACCEPT'
r_spd = safety_gate(tests[1][1], [c60])
assert r_spd['verdict'] == 'CLIP' and abs(r_spd['v'].max() - 60 * KMH) < 1e-9
r_acc = safety_gate(tests[2][1], [c60])
assert r_acc['verdict'] == 'CLIP' and np.max(np.abs(np.diff(r_acc['v']) / DT)) <= 4.0 + 1e-9
r_lft = safety_gate(tests[3][1], [c60, cLeft])
assert r_lft['verdict'] == 'REJECT' and np.all(np.diff(r_lft['v']) <= 1e-12), 'fallback 必须单调不增'
assert safety_gate(tr_roll, [cStop])['verdict'] == 'REJECT'
print('\n✅ CLIP 与 REJECT 的分界，正是第 1 节那张表里的 projectable 字段。')

In [ ]:
# —— 干预率：安全层每一次非 ACCEPT，都是一个自动标注好的 hard case（接 C58）——
rng = np.random.default_rng(7)
cons_urban = [c60, cLeft]
tally, rejected_frames = {'ACCEPT': 0, 'CLIP': 0, 'REJECT': 0}, []
for i in range(300):
    v0 = rng.uniform(9.0, 17.0)
    v = np.clip(v0 + np.cumsum(rng.normal(0, 0.15, 40)), 0.0, None)
    man = ['straight', 'left', 'right'][int(rng.choice(3, p=[0.80, 0.10, 0.10]))]
    r = safety_gate(make_traj(v, maneuver=man), cons_urban)
    tally[r['verdict']] += 1
    if r['verdict'] == 'REJECT':
        rejected_frames.append(i)

n = sum(tally.values())
print('300 帧 VLA 输出经过安全层：')
for k in ('ACCEPT', 'CLIP', 'REJECT'):
    print('  %-7s %4d  (%5.1f%%)' % (k, tally[k], 100.0 * tally[k] / n))
print('  **干预率** = (CLIP + REJECT) / N = %.1f%%' % (100.0 * (tally['CLIP'] + tally['REJECT']) / n))
assert n == 300 and tally['REJECT'] > 0 and tally['CLIP'] > 0 and tally['ACCEPT'] > 0
assert tally['REJECT'] == 33, '禁左轨迹约占 10%，全部被整条拒绝'
print('\n干预率的两个用法：')
print('  ① 它是 VLA 质量最直接的代理指标 —— 升高说明模型在系统性地输出不安全动作')
print('  ② 每一次 REJECT 都是一个**自动标注好的 hard case**，直接进数据闭环（C58 模块 03 的触发器）')
print('  ⚠️ 但它也是**能力上界**的度量：兜底太严 -> 频繁降级 -> 体验差，且 VLA 永远收不到「它错了」的信号。')

## 6 · 快慢双系统：留给思维链的时间到底有多少

从「必须在标志所在位置前完成动作」倒推：

$$T_{\text{reason}}^{\max} = \frac{d_{\text{detect}} - v^{2}/(2a_{\text{comf}})}{v} - T_{\text{perceive}} - T_{\text{act}}$$

代入 $a_{\text{comf}}=3\ \mathrm{m/s^2}$（乘客不会觉得难受的减速度）、
$T_{\text{perceive}}=0.15$ s、$T_{\text{act}}=0.10$ s。**结果里有负数，那是物理上就来不及。**

In [ ]:
A_COMF, T_PERC, T_ACT = 3.0, 0.15, 0.10

def reason_budget(v_kmh, d_detect, a_comf=A_COMF):
    v = v_kmh * KMH
    d_brake = v * v / (2 * a_comf)
    return (d_detect - d_brake) / v - T_PERC - T_ACT, d_brake

def px_size(obj_m, dist_m, width_px=1920, hfov_deg=60.0):
    """针孔模型：物体在图像上的像素尺寸（C57 的物理推导）"""
    f = (width_px / 2) / math.tan(math.radians(hfov_deg / 2))
    return f * obj_m / dist_m

SCEN = [('城区限速牌',   60,  80, 60.0), ('城区·检出晚',   60,  50, 60.0),
        ('高速·广角',   120, 150, 60.0), ('高速·长焦',    120, 250, 20.0),
        ('临时施工牌',    80,  60, 60.0)]
print(f"{'场景':<14s}{'车速 km/h':>10s}{'检出距离 m':>12s}{'制动距离 m':>12s}"
      f"{'可用推理时间 s':>15s}{'牌的像素尺寸':>14s}")
for name, vk, dd, fov in SCEN:
    b, db = reason_budget(vk, dd)
    px = px_size(0.6, dd, hfov_deg=fov)
    flag = '' if b > 0 else '  <- 来不及'
    print('%-14s%10d%12d%12.1f%15.2f%13.1f px%s' % (name, vk, dd, db, b, px, flag))

assert abs(reason_budget(60, 80)[0] - 1.7722) < 1e-3
assert reason_budget(60, 50)[0] < 0 and reason_budget(120, 150)[0] < 0
assert abs(reason_budget(120, 250)[0] - 1.6944) < 1e-3
assert abs(px_size(0.6, 250, hfov_deg=60.0) - 3.99) < 0.05, '250 m 外 60 cm 的牌在广角上只有 4 像素'
assert px_size(0.6, 250, hfov_deg=20.0) > 12.0, '长焦把它救回到 13 像素'
print('\n⚠️ 「高速场景的推理时间」看起来是软件问题，答案却在硬件上：')
print('   250 m 外 60 cm 的牌在 1920/60° 广角上只有 **4.0 像素** —— 任何检测器都做不到。')
print('   要么上长焦（20° FOV -> 13.1 像素），要么承认高速限速必须由**地图**提供、视觉只做校验。')

In [ ]:
# —— CoT 的延迟-准确率权衡：给定推理预算，最多能跑几个 token ——
def cot_latency(n_tok, base=0.20, per_tok=0.025):      # 40 tok/s
    return base + per_tok * n_tok

def cot_accuracy(n_tok, p0=0.80, p_max=0.99, tau=40.0):
    return p_max - (p_max - p0) * math.exp(-n_tok / tau)

def plan_reasoning(v_kmh, d_detect):
    b, _ = reason_budget(v_kmh, d_detect)
    if b < cot_latency(0):                              # 连 0 个 token 的固定开销都放不下
        return dict(budget_s=b, n_tok=0, acc=cot_accuracy(0), system='FAST')
    n = int((b - 0.20) // 0.025)
    return dict(budget_s=b, n_tok=n, acc=cot_accuracy(n), system='SLOW')

print(f"{'场景':<14s}{'可用推理 s':>12s}{'可跑 CoT token':>15s}{'预期正确率':>12s}{'走哪个系统':>12s}")
for name, vk, dd, _ in SCEN:
    p = plan_reasoning(vk, dd)
    print('%-14s%12.2f%15d%11.1f%%%12s' % (name, p['budget_s'], p['n_tok'], 100 * p['acc'], p['system']))
assert plan_reasoning(60, 80)['n_tok'] == 62 and plan_reasoning(60, 80)['system'] == 'SLOW'
assert plan_reasoning(60, 50)['system'] == 'FAST' and plan_reasoning(120, 150)['system'] == 'FAST'
assert plan_reasoning(60, 80)['acc'] > 0.94 > cot_accuracy(0)

# —— 推理预算随车速自适应（固定检出距离 120 m）——
print('\n固定检出距离 120 m，推理预算随车速自适应：')
print(f"{'车速 km/h':>10s}{'可用推理 s':>12s}{'CoT token 上限':>15s}{'预期正确率':>12s}{'系统':>8s}")
prev = 10 ** 9
for vk in [40, 60, 80, 100, 120, 140]:
    p = plan_reasoning(vk, 120.0)
    print('%10d%12.2f%15d%11.1f%%%8s' % (vk, p['budget_s'], p['n_tok'], 100 * p['acc'], p['system']))
    assert p['n_tok'] <= prev, '车速越高，允许的推理越短 —— 必须单调'
    prev = p['n_tok']
print('\n✅ 快慢双系统的关键不是「慢系统更聪明」，而是**慢系统输出约束、快系统输出控制量**：')
print('   「限速 60 从里程 1240 开始」这条陈述在 0.6 s 后依然正确，而一条轨迹在 0.6 s 后起点就错了。')

## ✏️ 练习 1：约束的纵向作用域

实现 `scope_of(det, ego_s)` 返回 `(begins_at_s, ends_at_s)`：
- `begins_at_s = ego_s + det['range_m']` —— **牌所在里程**，不是检出时的自车里程
- 若辅助牌给了 `det['plate']['range_m']`（「前方 2 km」），则 `ends_at_s = begins_at_s + 该值`；
  否则 `ends_at_s = float('inf')`

> 这道题只有两行，但它是 over-hold / early-release 两类事故的源头。
> 用「检出时的自车里程」当起点，作用域整体前移几十米——**提前失效是安全问题**。

In [ ]:
def scope_of(det, ego_s):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
d = dict(sign='speed_limit', value=60, conf=0.9, range_m=82.4, track_id=1)
assert scope_of(d, 1157.6) == (1240.0, float('inf')), scope_of(d, 1157.6)

d2 = dict(sign='speed_limit', value=40, conf=0.9, range_m=55.0, track_id=2,
          plate=dict(range_m=2000.0))
assert scope_of(d2, 1200.0) == (1255.0, 3255.0), scope_of(d2, 1200.0)

# 同一块牌在不同帧被检出，算出的作用域必须**完全一致**（这是最好的一致性自检）
spans = {scope_of(dict(sign='speed_limit', value=60, conf=0.9,
                       range_m=55.0 - s, track_id=47), s) for s in (15.0, 22.5, 30.0, 37.5)}
assert len(spans) == 1 and spans.pop() == (55.0, float('inf')), '作用域必须与观测时刻无关'

# 错误版本：用检出时的自车里程当起点 -> 作用域整体前移
wrong_begin = 1157.6
print('正确起点 %.1f m  vs  错误起点 %.1f m  ->  作用域整体前移 %.1f m'
      % (1240.0, wrong_begin, 1240.0 - wrong_begin))
print('✅ 练习 1 通过：约束的作用域必须与「什么时候看到它」无关。')

## ✏️ 练习 2：优先级消解

实现 `resolve_speed(cands)`，输入是一组形如
`dict(id=..., authority=int, specificity=int, begins_at_s=float, value=float)` 的限速候选，
返回 `(winner_id, value, tie)`：

1. 按字典序 `(authority, specificity, begins_at_s)` 取最大；
2. 若并列多个（三元组完全相同）→ `tie=True`，并在并列者中**取 value 最小的**（限速的保守方向是 MIN）；
3. 空输入返回 `(None, float('inf'), False)`。

In [ ]:
def resolve_speed(cands):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
C = lambda i, a, sp, s, v: dict(id=i, authority=a, specificity=sp, begins_at_s=s, value=v)

# ① 施工临时牌(4) > 固定牌(2) > 地图(1)
r = resolve_speed([C('t1', 4, 0, 1030., 11.11), C('f1', 2, 0, 1010., 22.22), C('h1', 1, 0, 1000., 27.78)])
assert r == ('t1', 11.11, False), r

# ② 同权威 -> 特异性高的赢（lex specialis）
r = resolve_speed([C('a', 2, 0, 100., 30.0), C('b', 2, 2, 100., 25.0)])
assert r == ('b', 25.0, False), r

# ③ 同权威同特异性 -> **牌所在里程更靠后的赢**（后遇到的胜）
r = resolve_speed([C('a', 2, 0, 100., 30.0), C('b', 2, 0, 180., 22.2)])
assert r == ('b', 22.2, False), r

# ④ 三元组完全相同 -> tie，取最保守（min value）
r = resolve_speed([C('g1', 2, 0, 2050., 27.8), C('g2', 2, 0, 2050., 22.2)])
assert r == ('g2', 22.2, True), r

# ⑤ 空输入 -> 无约束
assert resolve_speed([]) == (None, float('inf'), False)

# ⑥ 「后遇到的胜」比的是**牌所在里程**而不是检出时刻：远处的牌可能先被检出
far_seen_first  = C('far',  2, 0, 300., 33.3)     # 先检出，但牌在 300 m
near_seen_later = C('near', 2, 0, 150., 16.7)     # 后检出，牌在 150 m
wid, val, _ = resolve_speed([far_seen_first, near_seen_later])
assert wid == 'far', '按里程排序 -> 300 m 那块更靠后，它才是「后遇到的」'
print('✅ 练习 2 通过：消解是三元组字典序 + 一条显式的平票兜底，全程可写进决策日志。')

## ✏️ 练习 3：停车让行的时序判定

实现 `full_stop_ok(s, v, s_stop, tol=2.0, v_eps=0.10)` → `bool`：

必须存在某个采样点同时满足 `v <= v_eps` **且** `s_stop - tol <= s <= s_stop`。

三种必须判 FAIL 的情况：**rolling stop**（速度没到 0）、**停得太早**（离停止线太远）、
**冲过停止线才停**（`s > s_stop`）。

In [ ]:
def full_stop_ok(s, v, s_stop, tol=2.0, v_eps=0.10):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测（手算）——
s_ = np.array([10.0, 14.0, 17.0, 18.0, 18.0, 20.0])
v_ = np.array([12.0,  8.0,  3.0,  0.0,  0.0,  4.0])
assert full_stop_ok(s_, v_, 18.5) is True                     # 在 18.0 停下，线在 18.5，差 0.5 m
assert full_stop_ok(s_, v_, 30.0) is False                    # 停得太早（差 12 m > tol）
assert full_stop_ok(s_, v_, 17.0) is False                    # 冲过停止线 1 m 才停

v_roll_ = np.array([12.0, 8.0, 3.0, 1.2, 1.2, 4.0])
assert full_stop_ok(s_, v_roll_, 18.5) is False, 'rolling stop 是违章，不是「基本停住了」'

# 放宽 tol 可以让「停得太早」通过 —— 说明 tol 是一个**产品/法规参数**，不是随手填的数
assert full_stop_ok(s_, v_, 30.0, tol=12.0) is True

# 用第 4 节的两条完整轨迹再验一遍
assert full_stop_ok(tr_full['s'], tr_full['v'], 18.5) is True
assert full_stop_ok(tr_roll['s'], tr_roll['v'], 18.5) is False
print('✅ 练习 3 通过：「完全停止」= ∃t*: v(t*)<=ε ∧ s(t*)∈[s0-δ, s0]，'
      '三个条件缺一不可，而且它是**不可投影**的。')

## ✏️ 练习 4：推理预算求解器（快慢双系统）

实现 `plan(v_kmh, d_detect, a_comf=3.0)` 返回
`dict(budget_s=..., n_tok=..., acc=..., system=...)`：

1. `budget_s = (d_detect - v²/(2·a_comf))/v - 0.15 - 0.10`（`v` 是 m/s）
2. 若 `budget_s < 0.20`（连 0 个 token 的固定开销都放不下）→ `n_tok=0`，`system='FAST'`
3. 否则 `n_tok = int((budget_s - 0.20) // 0.025)`，`system='SLOW'`
4. `acc = 0.99 - 0.19·exp(-n_tok/40)`

In [ ]:
def plan(v_kmh, d_detect, a_comf=3.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
p = plan(60, 80)
assert abs(p['budget_s'] - 1.77222) < 1e-4 and p['n_tok'] == 62 and p['system'] == 'SLOW'
assert abs(p['acc'] - 0.94967) < 1e-4, p['acc']

assert plan(60, 50)['system'] == 'FAST' and plan(60, 50)['n_tok'] == 0
assert plan(120, 150)['budget_s'] < 0 and plan(120, 150)['system'] == 'FAST'
assert plan(120, 250)['n_tok'] == 59 and plan(120, 250)['system'] == 'SLOW'
assert abs(plan(60, 50)['acc'] - 0.80) < 1e-9, '快系统的准确率就是 n_tok=0 时的基线'

# 单调性：车速越高 -> 可跑的推理越短（固定检出距离）
toks = [plan(v, 120.0)['n_tok'] for v in (40, 60, 80, 100, 120, 140)]
assert toks == sorted(toks, reverse=True), toks
print('固定检出距离 120 m，各车速的 CoT token 上限:', toks)

# 反过来：给定想跑的 token 数，求最低的检出距离要求
def min_detect_distance(v_kmh, n_tok, a_comf=3.0):
    v = v_kmh / 3.6
    need_t = 0.20 + 0.025 * n_tok + 0.15 + 0.10
    return v * need_t + v * v / (2 * a_comf)

d_need = min_detect_distance(120, 60)
assert plan(120, d_need + 0.5)['n_tok'] >= 60, (d_need, plan(120, d_need + 0.5))
print('120 km/h 下要跑 60 个 CoT token，检出距离至少需要 %.1f m' % d_need)
print('   -> 60 cm 的牌在这个距离上只有 %.1f 像素（1920/60° 广角）'
      % px_size(0.6, d_need, hfov_deg=60.0))
print('✅ 练习 4 通过：「能不能开思维链」不是模型问题，是一道由车速、检出距离和相机 FOV 决定的算术题。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def scope_of(det, ego_s):
    begins = ego_s + det['range_m']            # **牌所在里程**
    span = det.get('plate', {}).get('range_m')
    return (begins, begins + span if span else float('inf'))

In [ ]:
# 练习 2 参考答案
def resolve_speed(cands):
    if not cands:
        return None, float('inf'), False
    key = lambda c: (c['authority'], c['specificity'], c['begins_at_s'])
    best = max(key(c) for c in cands)
    top = [c for c in cands if key(c) == best]
    tie = len(top) > 1
    win = min(top, key=lambda c: c['value']) if tie else top[0]
    return win['id'], win['value'], tie

In [ ]:
# 练习 3 参考答案
def full_stop_ok(s, v, s_stop, tol=2.0, v_eps=0.10):
    s = np.asarray(s, float); v = np.asarray(v, float)
    return bool(np.any((v <= v_eps) & (s >= s_stop - tol) & (s <= s_stop)))

In [ ]:
# 练习 4 参考答案
def plan(v_kmh, d_detect, a_comf=3.0):
    v = v_kmh / 3.6
    budget = (d_detect - v * v / (2 * a_comf)) / v - 0.15 - 0.10
    if budget < 0.20:
        n = 0
    else:
        n = int((budget - 0.20) // 0.025)
    return dict(budget_s=budget, n_tok=n,
                acc=0.99 - 0.19 * math.exp(-n / 40.0),
                system='SLOW' if n > 0 else 'FAST')

---
## 🧪 真实工程胶囊：约束层的落地清单与决策日志 schema

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 交通规则约束层 · 落地清单（按顺序做，每一步都有明确的通过条件）
# ══════════════════════════════════════════════════════════════════════

# ① 约束对象的最小 schema（写进 proto/idl，版本化）
#    Constraint {
#      string  id;              // 全局唯一，日志与归因靠它
#      enum    kind;            // SPEED_MAX | SPEED_MIN | BAN_MANEUVER | FULL_STOP | REVOKE
#      string  arg;             // BAN_MANEUVER 的机动类型
#      double  value;           // **SI 单位（m/s），不要在接口上传 km/h**
#      enum    source;          // POLICE|TEMPORARY|VMS|FIXED_SIGN|HDMAP|DEFAULT
#      int32   authority;       // 由 source 决定，冗余存一份便于排序
#      int32   specificity;     // 辅助牌限定符个数
#      double  begins_at_s;     // **牌所在里程** = ego_s + range_m
#      double  ends_at_s;       // 辅助牌范围；无则 +inf
#      repeated string lanes;   // 横向作用域
#      string  vehicle_class;   // 车型作用域
#      double  tw_start, tw_end;// 时间窗
#      repeated string end_rules;   // END_SIGN|JUNCTION|ROAD_CLASS_CHANGE|...
#      bool    projectable;     // **安全层据此决定 CLIP 还是 REJECT**
#      enum    conservatism;    // MIN|MAX|UNION —— 别在代码里到处硬编码 min()
#      enum    state;           // TENTATIVE|CONFIRMED|ACTIVE|EXPIRED|REVOKED|...
#      double  t_obs; double conf; repeated int64 evidence_track_ids;
#    }
# 通过条件：下游任意模块拿到一条 Constraint，无需回查感知就能独立判断它是否适用于自己

# ② 五条失效路径必须各有一条单元测试（少一条 = 一整类线上 bug）
#    [ ] END_SIGN         解除牌被 CONFIRM -> 匹配约束 REVOKED
#    [ ] JUNCTION         通过路口 -> 限速类 EXPIRED
#    [ ] ROAD_CLASS_CHANGE 驶出匝道 -> 匝道限速 EXPIRED   ← 「匝道限速带上高速」
#    [ ] RANGE_EXHAUSTED  辅助牌「前方 2km」走完 -> EXPIRED
#    [ ] SUPERSEDED       同槽位更靠后的新约束顶掉旧的
#    另加运行时不变式（**放进生产代码，不只是测试**）：
#      assert 同一 (kind, arg, lanes, source) 槽位上 ACTIVE 约束数 <= 1

# ③ 两类失效错误分别度量，门禁不同
#    over_hold_m      = 应失效点之后仍 ACTIVE 的里程    -> 体验指标，p95 <= 30 m
#    early_release_m  = 应失效点之前就退出的里程        -> **安全指标，硬门禁 = 0 次**
#    tie_rate         = 消解平票率                      -> 健康度，突增 = 车道关联坏了
#    intervention_rate= 安全层 (CLIP+REJECT)/N          -> VLA 质量代理 + hard case 来源

# ④ 决策日志（每帧一条，采样落盘；事故 3 个月后只看日志要能重建决策链）
#    { t_ego, ego_s, ego_v,
#      active_constraints: [...],      // **全部 ACTIVE，不只是获胜的那条**
#      binding_constraint: "c17",
#      resolution_trace:  [(id, key_tuple, verdict, tie_flag)],
#      vla_raw_action:    [...],       // 未经安全层
#      safety_verdict:    ACCEPT|CLIP|REJECT,
#      safety_reasons:    [...],
#      final_action:      [...],       // vla_raw 与 final 的差 = 干预量
#      latency_ms: {perceive, reason, constrain, safety, total} }

# ⑤ 安全层的四条独立性硬要求（缺一条，安全论证不成立）
#    [ ] 不同代码库/不同语言，不 link VLA 的任何库
#    [ ] **不同输入路径** —— 不能只吃 VLA 用过的那份感知结果（否则 common-mode failure）
#    [ ] 独立算力与时间预算，VLA 超时不能拖垮它
#    [ ] 不同团队开发与评审，变更走更严格流程

# ⑥ 思维链的三条硬约束（延迟是数据依赖的，套路同 NMS：设上限 + 备降级）
#    [ ] max_reasoning_tokens 硬上限，超时立即用快系统结果并记 timeout 事件
#    [ ] 推理预算随车速自适应：n_max = ((d_detect - v^2/2a)/v - 0.25 - 0.20) / t_per_token
#    [ ] 投机执行：快系统结果先下发，慢系统结果到达后再修正

# ⑦ TSR 专项检查
#    [ ] 解除牌的召回率**单独统计**（它比主标志难检，却决定了 over-hold）
#    [ ] 对向车道/辅路标志：用「牌的朝向」做几何过滤，而不是指望分类器
#    [ ] 货车/时段辅助牌必须能被识别**然后忽略**（识别不出来会误执行）
#    [ ] 限速值只做上界，最终速度 = min(法定, 天气降级, 曲率 sqrt(a_lat/kappa), 跟车)
'''
print(RECIPE)
for token in ['projectable', 'conservatism', 'early_release_m', 'intervention_rate',
              'ROAD_CLASS_CHANGE', 'common-mode', 'max_reasoning_tokens', '朝向']:
    assert token in RECIPE, token
print('✅ 清单覆盖：schema / 五条失效路径 / 双指标门禁 / 决策日志 / 安全层独立性 / CoT 预算 / TSR 专项')

### 小结

- **标志 → 动作要经过三级翻译**：感知语义（观测）→ 规范语义（法规解释）→ 可执行约束（对象）。
  量产里标志相关的线上问题，**多数根因不在识别，而在这层翻译**。
- **约束的正确性 = 值对 × 从哪开始对 × 到哪结束对 × 对谁生效对。**
  失效有五条路（解除牌 / 路口 / 道路等级变化 / 范围耗尽 / 被取代），少实现一条就是一整类 bug；
  notebook 里「匝道限速带上高速」在缺地图信号时 over-hold **75 米 / 5 秒**。
  `CONFIRMED` 与 `ACTIVE` 必须分开——中间隔着一整个检出距离。
- **消解是字典序三元组 (authority, specificity, begins_at_s)**，平票取最保守 **并计数**。
  但**「保守 = 安全」是错的**：最低限速牌的保守方向相反；施工区一直保持 40 会被追尾。
  保守方向要写成约束的字段，不能散落成一堆 `min()`。
- **可投影性决定安全层的处置**：速度/加速度是凸区间 → **CLIP**；
  禁行机动是不连通集、停车让行是时序存在量词 → **REJECT + 规则型 fallback**。
- **快慢双系统的关键是「慢系统输出约束、快系统输出控制量」**：约束抗延迟，轨迹不抗延迟。
  可用推理时间 $=(d_{\text{detect}}-v^2/2a)/v-0.25$ s：60 km/h + 80 m 检出 = **1.77 s（约 62 个 token）**；
  120 km/h + 150 m 检出 = **负数**，物理上就来不及，必须靠地图。
  而 250 m 外 60 cm 的牌在广角上只有 **4 像素**——软件问题的答案在硬件上。
- **红线：VLA 的输出没有任何一条可以直接下发。**安全层要独立于 VLA（不同代码库、
  **不同输入路径**、独立算力、不同团队），做五项校验、三态输出。
  **干预率既是 VLA 质量的代理指标，也是自动标注好的 hard case 来源。**

下一站：**模块 05 · VLA 的评测与上车** —— 开环评测为什么会严重高估，以及怎么把车真的开上路。